# ThinkGuard 4단계: bge-m3 판단 모델 학습 (Colab T4)

**준비**: Google Drive `내 드라이브/ThinkGuard_model/`에 아래 파일이 있어야 한다.
```
scripts/common.py, scripts/03_baseline_tfidf.py, scripts/04_train_bge_m3.py
data/joined/*.jsonl          (영어)
data/ko/*.jsonl              (한국어 번역, 번역 노트북 결과)
```
런타임은 **T4 GPU**로 바꾼다. 번역 노트북이 같은 GPU를 쓰는 중이면 번역이 끝난 뒤에 실행한다.

예상 시간 (T4, 3 epoch): 영어만 약 1~1.5시간, 영어+한국어 약 2~3시간.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/ThinkGuard_model'
%cd {PROJECT}
!ls scripts data
!wc -l data/joined/*.jsonl data/ko/*.jsonl

In [ ]:
!pip install -q "transformers>=4.45,<5" scikit-learn sentencepiece

## 1. 코드 점검 (1~2분)
작은 모델로 64행만 학습해서 오류가 없는지 본다. 점수는 의미 없다.

In [ ]:
!cd scripts && python 04_train_bge_m3.py --train-dirs ../data/joined ../data/ko --model intfloat/multilingual-e5-small --limit 64 --epochs 1 --max-len 128 --out-dir /content/smoke

## 2. 본 학습: 영어 + 한국어
- `--grad-checkpointing`: T4 메모리(15GB)에 맞추려고 켠다.
- `--save-model`: 가장 좋은 epoch의 모델을 Drive에 저장한다 (약 2.3GB).
- 결과: `results/bge_m3/<tag>/metrics.json`. test 점수가 영어(en)와 한국어(ko)로 나뉘어 출력된다.
- 메모리 부족(CUDA out of memory)이 나면 `--batch-size 4 --grad-accum 8`로 바꾼다.

In [ ]:
!cd scripts && python 04_train_bge_m3.py --train-dirs ../data/joined ../data/ko --view full --max-len 512 --batch-size 8 --grad-accum 4 --epochs 3 --grad-checkpointing --save-model --out-dir ../results/bge_m3 --tag en_ko_full

## 3. (비교 실험) 현재 발화만 입력
TF-IDF에서는 현재 발화만 넣는 쪽이 더 좋았다. bge-m3에서도 그런지 비교한다. 모델은 저장하지 않는다.

In [ ]:
!cd scripts && python 04_train_bge_m3.py --train-dirs ../data/joined ../data/ko --view message --max-len 512 --batch-size 8 --grad-accum 4 --epochs 3 --grad-checkpointing --out-dir ../results/bge_m3 --tag en_ko_message

## 4. TF-IDF 기준선도 같은 데이터로 (비교표용, CPU로 몇 분)

In [ ]:
!cd scripts && python 03_baseline_tfidf.py --data-dirs ../data/joined ../data/ko --tag en_ko --out-dir ../results/baseline_tfidf

## 5. 결과 비교

In [ ]:
import json, glob
for p in sorted(glob.glob('results/bge_m3/*/metrics.json')):
    m = json.load(open(p))
    for lang, r in m['test'].items():
        print(f"{p.split('/')[-2]:16s} test[{lang}] request F1 {r['requests_complete_answer']['f1']:.3f} | "
              f"attempt F1 {r['has_self_attempt_evidence']['f1']:.3f} | learning=false F1 {r['is_learning_context']['f1_false_class']:.3f} | "
              f"false_unlock {r['false_unlock']['count']}/{r['false_unlock']['of']}")